# Dataset Prefetch

- HF `dataset` load FineWeb-Edu 100BT
- consumes approx 750GB disk space to download and extract!!

Note:

- Karpathy converts dataset into compressed parquet shards
- I am using raw HF dataset and eating the 750GB for now

In [ ]:
import datasets

/home/user/projects/my-nanochat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_hf_path = "HuggingFaceFW/fineweb-edu"
dataset_hf_name = "sample-100BT"
dataset_hf_split = "train"

In [3]:
dataset_hf_names = datasets.get_dataset_config_names(dataset_hf_path)
print(f"Dataset configs: {dataset_hf_names[:5]}")
assert dataset_hf_name in dataset_hf_names

Dataset configs: ['default', 'sample-10BT', 'sample-100BT', 'sample-350BT', 'CC-MAIN-2025-05']


In [4]:
dataset_hf_splits = datasets.get_dataset_split_names(dataset_hf_path, dataset_hf_name)
print(f"Dataset splits: {dataset_hf_splits}")
assert dataset_hf_split in dataset_hf_splits

Dataset splits: ['train']


In [ ]:
# NOTE: Takes ~750GB disk space!!!
dataset = datasets.load_dataset(
    dataset_hf_path,
    name=dataset_hf_name,
    split=dataset_hf_split,
    # streaming=True,
)
dataset = dataset.shuffle(seed=42)  # Match nanochat repackage_data_reference.py seed

In [ ]:
# Print some examples matching Karpathy's sharding scheme
row_group_size = 1024                 # Karpathy packs data in 1024-document row groups
characters_per_shard = 250_000_000    # He uses ~250M characters per shard

total_chars = 0
doc_in_group_idx = 0
group_idx = 0
shard_idx = 0

num_printed = 0

for i, example in enumerate(dataset):
    text = example["text"]
    num_chars = len(text)

    if group_idx == 0 and doc_in_group_idx in (0, row_group_size - 1):
        print(f"{i:6d}:{shard_idx:4d}:{group_idx:4d}:{doc_in_group_idx:4d}: {text[:10]!r} ... {text[-10:]!r} (len={num_chars})")
        num_printed += 1

    total_chars += num_chars
    doc_in_group_idx += 1

    if doc_in_group_idx % row_group_size == 0:
        doc_in_group_idx = 0
        group_idx += 1
        if total_chars >= characters_per_shard:
            total_chars = 0
            group_idx = 0
            shard_idx += 1
    
    if num_printed >= 10:
        break

     0:   0:   0:   0: 'Shipment &' ... 'al impact.' (len=8657)
  1023:   0:   0:1023: '“Human pop' ... ' like one.' (len=3136)
 53248:   1:   0:   0: 'How to run' ... ' reserved.' (len=978)
 54271:   1:   0:1023: 'Textbooks ' ... 'ce system.' (len=9009)
106496:   2:   0:   0: 'Recycling ' ... 'ample.com.' (len=6257)
107519:   2:   0:1023: 'The Ancien' ... 'lic Online' (len=4182)
159744:   3:   0:   0: 'Chapter 1 ' ... 's - Part 2' (len=4736)
160767:   3:   0:1023: 'Research h' ... ' remedies.' (len=3795)
212992:   4:   0:   0: 'Franklin H' ... ' rarities.' (len=2241)
214015:   4:   0:1023: 'NEW YORK (' ... 'gagement.”' (len=5369)


# Validate against NanoChat

In [28]:
import pyarrow.parquet as pq

In [ ]:
# Requires running nanochat speedrun.sh
# Especially nanochat script: python -m nanochat.dataset -n 240
for shard2 in range(5):
    pf = pq.ParquetFile(f"/home/user/.cache/nanochat/base_data/shard_{shard2:05d}.parquet")
    group_idx2 = 0
    rg = pf.read_row_group(group_idx2)
    batch = rg.column('text').to_pylist()
    text = batch[0]
    print(f"{shard2:4d}:{group_idx2:4d}: {text[:10]!r} ... {text[-10:]!r} (len={len(text)})")
    text = batch[-1]
    print(f"{shard2:4d}:{group_idx2:4d}: {text[:10]!r} ... {text[-10:]!r} (len={len(text)})")

   0:   0: 'Shipment &' ... 'al impact.' (len=8657)
   0:   0: '“Human pop' ... ' like one.' (len=3136)
   1:   0: 'How to run' ... ' reserved.' (len=978)
   1:   0: 'Textbooks ' ... 'ce system.' (len=9009)
   2:   0: 'Recycling ' ... 'ample.com.' (len=6257)
   2:   0: 'The Ancien' ... 'lic Online' (len=4182)
   3:   0: 'Chapter 1 ' ... 's - Part 2' (len=4736)
   3:   0: 'Research h' ... ' remedies.' (len=3795)
   4:   0: 'Franklin H' ... ' rarities.' (len=2241)
   4:   0: 'NEW YORK (' ... 'gagement.”' (len=5369)
